# Assignment 3
## CTC decoding augmented with a language model

- The assignment is due by **July 21 (Tuesday) 23:59**
- Write/implement your solutions in a copy of this notebook **without changing the structure**
- Finally submit in a **PDF** format (with proper outputs) by following `File -> Save and Export Notebook As -> PDF` and then upload
- The assignment has **5** questions in total scattered in this notebook

## Install Requirements
- The following packages are set to work for GPU. If you intend to run on CPU (not recommended), you have to figure out the right versions yourself.

In [ ]:
!pip install torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
!pip install "transformers[torch]" "datasets" "torchcodec<0.5" "pyctcdecode" "num2words" "librosa" "evaluate" "jiwer" --upgrade

In [ ]:
!conda install -c conda-forge ffmpeg=4.4 -y

In [ ]:
!conda install -c conda-forge compilers cmake make -y
!pip install https://github.com/kpu/kenlm/archive/master.zip

## Load and preprocess dataset

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset(
    "google/fleurs",
    "de_de",
    streaming=False
)

dataset = dataset.cast_column(
    "audio",
    Audio(decode=False)
)

> Let's create a new column `processed_text` across all datasets by replacing numerals with text following the procedure given in the previous assignment (2). We do not perform umlaut correction as umlaut characters are supported by the model we are going to use here.

In [ ]:
import re
from num2words import num2words


def replace_number(match):
    token = match.group(0)

    # Ordinal (e.g. 3., 21., 100.)
    if token.endswith("."):
        value = int(token[:-1])
        return num2words(value, lang="de", to="ordinal")

    value = int(token)

    # Years between 1100 and 2050
    if 1100 < value < 2050:
        return num2words(value, lang="de", to="year")

    # Cardinal
    return num2words(value, lang="de")

re.sub(r"\d+\.|\d+", replace_number, "Ich wurde am 3. Mai 1985 geboren.")

### Q1

Fill in the code

In [ ]:
def preprocess_text(example):
    ## Your code here


dataset = dataset.map(preprocess_text)

## Train a Language Model

### Q2

Prepare corpus iterating through `dataset['train']` and save it to `corpus.txt` with one sentence per line, each sentence with space separated words. 

In [ ]:
## Your code here


### Q3

- Install KenLM on your computer by following [https://github.com/kpu/kenlm](https://github.com/kpu/kenlm).
- Train a trigram language model on this corpus using the command
  > `lmplz --order 3 --discount_fallback --text corpus.txt --arpa lm.arpa --memory 10G`

## Compare WER using CTC with and without LM

Let's run `aware-ai/wav2vec2-base-german` on sample audio from `datasets['validation']` and see its output

In [ ]:
import torch
from IPython.display import Audio
import librosa
import io
from transformers import AutoProcessor, AutoModelForCTC


# Get one validation sample
sample = dataset["validation"][0]

audio_bytes = sample["audio"]["bytes"]

# Play audio
waveform, sr = librosa.load(
    io.BytesIO(audio_bytes),
    sr=None,
    mono=True
)

Audio(waveform, rate=sr)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCTC

processor = AutoProcessor.from_pretrained(
    "aware-ai/wav2vec2-base-german"
)

model = AutoModelForCTC.from_pretrained(
    "aware-ai/wav2vec2-base-german"
)

model.eval()

inputs = processor(
    waveform,
    sampling_rate=sr,
    return_tensors="pt",
    padding=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

pred_ids = torch.argmax(logits, dim=-1)

prediction = processor.batch_decode(pred_ids)[0]

print("Reference:")
print(sample["processed_text"])

print("\nPrediction:")
print(prediction)

> Let's write an evaluator function that takes predictions and transcriptions to report word error rate (WER) and character error rate (CER)

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def evaluate_asr(predictions, references):
    """
    Parameters
    ----------
    predictions : list[str]
        ASR hypotheses
    references : list[str]
        Ground-truth transcriptions

    Returns
    -------
    dict
        {"WER": ..., "CER": ...}
    """
    wer = wer_metric.compute(
        predictions=predictions,
        references=references
    )

    cer = cer_metric.compute(
        predictions=predictions,
        references=references
    )

    return {
        "WER": wer,
        "CER": cer,
    }

### Q4

Let's run predictions on plain model (as in the example) and report WER and CER. Fill in the missing code

In [ ]:
from tqdm.auto import tqdm

predictions = []
references = []

for sample in tqdm(dataset["test"]):
    ### Your code here
    
    predictions.append(prediction)
    references.append(sample["processed_text"])

results = evaluate_asr(
    predictions,
    references
)

print(f"WER: {results['WER']:.4f}")
print(f"CER: {results['CER']:.4f}")

> Now use the trained `lm.arpa` via the module `pyctcdecode` to improve decodings and report WER and CER on `dataset[test]`

- Given CTC probabilities $p_{ctc}$ on input features $X = x_1, \dots, x_T$ and n-gram language model probabilities $p_{lm}(w_i \mid w_{i-1}, \ldots, w_{i-n+1})$ on corresponding word sequence $W = w_1, \ldots, w_L$ and alignments $\mathcal{A}(W)$, the decoding maximizes:

> $\mathcal{L}_{ctc+lm} = \sum_{\pi \in \mathcal{A}(W)}p_{ctc}(\pi \mid X) + \alpha\sum_{i=n}^L\log p_{lm}(w_i \mid w_{i-1},\ldots, w_{i-n+1}) + \beta |W|$

- In other words, $\alpha$ weighs the language model while $\beta$ is word insertion bonus

In [ ]:
from pyctcdecode import build_ctcdecoder

vocab = processor.tokenizer.get_vocab()

# id -> token order expected by pyctcdecode
labels = [
    token
    for token, idx in sorted(vocab.items(), key=lambda x: x[1]) if token[0] != '{'
]

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path="lm.arpa",
    alpha=0.2,
    beta=0.8
)

In [ ]:
sample = dataset["validation"][0]

waveform, sr = librosa.load(
    io.BytesIO(sample["audio"]["bytes"]),
    sr=None,
    mono=True
)

inputs = processor(
    waveform,
    sampling_rate=sr,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    logits = model(**inputs).logits

# Greedy decoding
pred_ids = torch.argmax(logits, dim=-1)

greedy_text = processor.batch_decode(
    pred_ids,
    skip_special_tokens=True
)[0]

# LM decoding
logits_np = logits[0].cpu().numpy()

lm_text = decoder.decode(logits_np, beam_width=100)

print("REFERENCE:")
print(sample["processed_text"])

print("\nGREEDY:")
print(greedy_text)

print("\nKENLM:")
print(lm_text)

### Q5

Write the code for evaluating CTC + LM decoding using `beam_width = 100` and report WER and CER. What are your observations?

In [ ]:
predictions = []
references = []

for sample in tqdm(dataset["test"]):

    ## Your code here

    predictions.append(prediction)
    references.append(sample["processed_text"])

results = evaluate_asr(
    predictions,
    references
)

print(f"WER: {results['WER']:.4f}")
print(f"CER: {results['CER']:.4f}")